# 1. Digital Twin Generator (Stream Simulator)

Let’s begin the implementation of **Project A: Telemetry Generator GUI**. Below is a clean, modular breakdown of the folder structure we defined, now expanded with **file-level implementation guidance**. This will help you scaffold the project and begin coding immediately.

---

# 🧩 Project A — Telemetry Generator GUI  
### 📁 `generator/`  
Main folder for simulation, GUI, and orchestration.

---

## **1. GUI Layer (`gui/`)**  
Handles all user interface components using PySide6 or PyQt6.

### 📁 `gui/`

| File | Description |
|------|-------------|
| `main_window.py` | Initializes the main window, assembles panels, connects signals. Uses `QMainWindow`, `QVBoxLayout`, `QSplitter`. |
| `schema_panel.py` | Contains 15 checkboxes grouped into 3 categories. Emits selected schema as a dictionary. |
| `settings_panel.py` | Contains row count spinbox, file format radio buttons, file size slider. Emits config object. |
| `preview_panel.py` | Displays a live Matplotlib or Plotly chart. Updates every few seconds with simulated data. |
| `status_bar.py` | Shows progress (rows written, file size), alerts (e.g., “Generation complete”), and socket status. |

---

## **2. Core Logic (`core/`)**  
Handles simulation, data generation, and writing.

### 📁 `core/`

| File | Description |
|------|-------------|
| `generator.py` | Main loop. Accepts config, schema, and starts chunked generation. Pushes preview data and writes to disk. |
| `column_models.py` | Contains simulation functions for each column type (e.g., `simulate_temperature()`, `simulate_rpm()`). Uses NumPy, SciPy, SymPy. |
| `writer.py` | Writes data to CSV or Parquet using Pandas or Polars. Supports chunking and compression. |
| `config_writer.py` | Writes `config.json` with schema, sampling rate, file path. Used by Analyzer. |
| `alert_socket.py` | Sends JSON alerts via local socket (e.g., `chunk_written`, `generation_complete`). Optional but modular. |

---

## **3. Utilities (`utils/`)**

### 📁 `utils/`

| File | Description |
|------|-------------|
| `file_tracker.py` | Tracks current file size and row count. Used to update progress bar and enforce limits. |
| `sampling_timer.py` | Controls simulation timing (e.g., 10 Hz). Can use `QTimer` or `threading.Timer`. |
| `preview_buffer.py` | Stores last N seconds of simulated data for live preview. Circular buffer or deque. |

---

## **4. Entry Point**

### 📄 `app.py`

| Purpose | Description |
|--------|-------------|
| Launch GUI | Initializes QApplication, loads `MainWindow`, connects signals, starts event loop. |

---

## ✅ Suggested Startup Flow

1. `app.py` launches `MainWindow`  
2. `MainWindow` assembles:
   - `SchemaPanel`
   - `SettingsPanel`
   - `PreviewPanel`
   - `StatusBar`
3. On “Generate”:
   - Collect schema + config
   - Start `Generator` thread
   - Generator:
     - Simulates data
     - Sends preview samples
     - Writes chunks to disk
     - Updates progress
     - Sends alerts via socket
     - Writes `config.json`

---

## 🧠 Optional Enhancements Already Integrated

| Feature | File(s) |
|--------|---------|
| Live preview | `preview_panel.py`, `preview_buffer.py` |
| Shared config file | `config_writer.py` |
| Alert socket | `alert_socket.py` |
| Progress bar | `status_bar.py`, `file_tracker.py` |

---


# 2. GUI folder

let’s begin implementing **Project A** in a clean, modular, and scalable way.
Below is the full implementation of pythonic files of the GUI folder, followed by a **thorough explanation** of their purpose, structure, inputs, outputs, and how it orchestrates the entire GUI.


## 2.1. main_window.py

Below is the full implementation of **`main_window.py`**, followed by a **thorough explanation** of its purpose, structure, inputs, outputs, and how it orchestrates the entire GUI.

This file is the *central hub* of the Generator GUI.

---

# 📄 `main_window.py`  
### **Full Implementation (PySide6)**  
This is a clean, production‑ready scaffold — no placeholder fluff, but fully structured for expansion.

```python
# generator/gui/main_window.py

from PySide6.QtWidgets import (
    QMainWindow, QWidget, QVBoxLayout, QHBoxLayout, QSplitter
)
from PySide6.QtCore import Qt

from .schema_panel import SchemaPanel
from .settings_panel import SettingsPanel
from .preview_panel import PreviewPanel
from .status_bar import StatusBar

from ..core.generator import TelemetryGenerator


class MainWindow(QMainWindow):
    """
    Main GUI window for the Telemetry Generator.
    Assembles all panels, manages layout, and orchestrates
    communication between GUI widgets and the generator backend.
    """

    def __init__(self):
        super().__init__()

        self.setWindowTitle("Telemetry Data Generator")
        self.setMinimumSize(1200, 700)

        # --- Panels ---
        self.schema_panel = SchemaPanel()
        self.settings_panel = SettingsPanel()
        self.preview_panel = PreviewPanel()
        self.status_bar = StatusBar()

        # --- Generator backend (initialized later) ---
        self.generator = None

        # --- Layout ---
        self._build_layout()

        # --- Connections ---
        self._connect_signals()

    # ---------------------------------------------------------
    # Layout Assembly
    # ---------------------------------------------------------
    def _build_layout(self):
        """
        Creates the main layout using a horizontal splitter:
        [Schema Panel | Settings Panel | Live Preview Panel]
        """
        central_widget = QWidget()
        main_layout = QVBoxLayout()

        splitter = QSplitter(Qt.Horizontal)
        splitter.addWidget(self.schema_panel)
        splitter.addWidget(self.settings_panel)
        splitter.addWidget(self.preview_panel)

        splitter.setSizes([300, 300, 600])  # initial proportions

        main_layout.addWidget(splitter)
        main_layout.addWidget(self.status_bar)

        central_widget.setLayout(main_layout)
        self.setCentralWidget(central_widget)

    # ---------------------------------------------------------
    # Signal Connections
    # ---------------------------------------------------------
    def _connect_signals(self):
        """
        Connects GUI events to backend logic.
        """
        self.settings_panel.generate_clicked.connect(self._start_generation)

    # ---------------------------------------------------------
    # Generation Logic
    # ---------------------------------------------------------
    def _start_generation(self):
        """
        Triggered when the user presses the 'Generate' button.
        Collects schema + settings, initializes the generator backend,
        and starts the simulation loop.
        """
        schema = self.schema_panel.get_schema()
        config = self.settings_panel.get_config()

        # Initialize backend generator
        self.generator = TelemetryGenerator(
            schema=schema,
            config=config,
            preview_callback=self.preview_panel.update_preview,
            progress_callback=self.status_bar.update_progress,
            alert_callback=self.status_bar.show_alert
        )

        self.status_bar.show_message("Starting data generation...")
        self.generator.start()

    # ---------------------------------------------------------
    # Cleanup
    # ---------------------------------------------------------
    def closeEvent(self, event):
        """
        Ensures generator thread stops cleanly when window closes.
        """
        if self.generator:
            self.generator.stop()
        event.accept()
```

---

# 🧠 **Detailed Explanation of `main_window.py`**

This file is the **central orchestrator** of the entire GUI.  
It does not simulate data, write files, or generate plots — instead, it:

- assembles the GUI layout  
- connects user actions to backend logic  
- manages the lifecycle of the generator thread  
- routes preview and progress updates to the correct widgets  

Let’s break it down.

---

# 🧩 **Purpose of `MainWindow`**

The `MainWindow`:

- is the **root window** of the application  
- contains all GUI panels  
- initializes the backend generator  
- connects signals (button clicks) to actions  
- handles cleanup when the app closes  

It is the “brain” that ties GUI and backend together.

---

# 🧱 **Structure Overview**

### **Panels included:**

| Panel | Purpose |
|-------|---------|
| `SchemaPanel` | User selects which telemetry columns to generate |
| `SettingsPanel` | User configures file size, row count, format |
| `PreviewPanel` | Shows live simulated data |
| `StatusBar` | Shows progress, alerts, messages |

These are arranged in a **horizontal splitter** for flexible resizing.

---

# 🔌 **Signal Connections**

The key connection:

```python
self.settings_panel.generate_clicked.connect(self._start_generation)
```

This means:

- When the user presses **Generate**,  
- The main window collects schema + settings,  
- Creates a `TelemetryGenerator` instance,  
- Starts the simulation loop.

---

# 🚀 **Generation Flow**

When the user clicks **Generate**:

1. `schema_panel.get_schema()`  
   → returns list of selected columns

2. `settings_panel.get_config()`  
   → returns a config object with:
   - file path  
   - file format  
   - target size  
   - sampling rate  
   - chunk size  

3. `TelemetryGenerator` is created with:
   - schema  
   - config  
   - callbacks for preview, progress, alerts  

4. `generator.start()`  
   → begins simulation in a background thread

5. `preview_panel.update_preview()`  
   → receives small batches of simulated data

6. `status_bar.update_progress()`  
   → updates progress bar and file size

7. `status_bar.show_alert()`  
   → displays alerts (e.g., “Chunk written”, “Generation complete”)

---

# 🧹 **Cleanup Logic**

When the window closes:

```python
if self.generator:
    self.generator.stop()
```

This ensures:

- no orphan threads  
- no file corruption  
- no socket left open  

---

# 📥 Inputs

The `MainWindow` receives:

- user interactions (checkboxes, sliders, buttons)
- schema from `SchemaPanel`
- config from `SettingsPanel`

---

# 📤 Outputs

The `MainWindow` produces:

- a running generator thread  
- live preview updates  
- progress updates  
- alerts  
- clean shutdown  

---

# 🎯 Summary

`main_window.py` is the **central coordinator** of the Generator GUI.  
It:

- builds the layout  
- connects GUI to backend  
- starts/stops the generator  
- routes preview and progress updates  
- ensures clean shutdown  

It is the backbone of Project A.

---


## 2.2. schema_panel.py

Here is a clean, production‑ready implementation of **`schema_panel.py`**, followed by a thorough explanation of its purpose, structure, inputs, outputs, and how it integrates into the Generator GUI.

This file is the **schema builder** of Project A: it defines the 15 telemetry columns and exposes a simple API (`get_schema()`) for the rest of the system.

---

# 📄 `schema_panel.py`  
### **Full Implementation (PySide6)**

```python
# generator/gui/schema_panel.py

from PySide6.QtWidgets import (
    QWidget, QVBoxLayout, QGroupBox, QCheckBox, QLabel
)
from PySide6.QtCore import Qt


class SchemaPanel(QWidget):
    """
    Panel containing the 15 predefined telemetry column checkboxes.
    Columns are grouped into:
        - Numeric Sensors
        - Categorical / Boolean
        - Auxiliary (timestamp, logs, cycle counter)

    Provides:
        get_schema() -> list of dicts describing selected columns
    """

    def __init__(self):
        super().__init__()

        self.setMinimumWidth(280)
        self._build_ui()

    # ---------------------------------------------------------
    # UI Construction
    # ---------------------------------------------------------
    def _build_ui(self):
        layout = QVBoxLayout()

        # --- Numeric Sensors ---
        numeric_group = QGroupBox("Numeric Sensors")
        numeric_layout = QVBoxLayout()
        self.cb_temperature = QCheckBox("Temperature")
        self.cb_rpm = QCheckBox("Motor RPM")
        self.cb_vibration = QCheckBox("Vibration Level")
        self.cb_power = QCheckBox("Power Consumption")
        self.cb_voltage = QCheckBox("Voltage")
        self.cb_current = QCheckBox("Current")
        self.cb_pressure = QCheckBox("Pressure / Load")
        self.cb_noise = QCheckBox("Noise Level")

        for cb in [
            self.cb_temperature, self.cb_rpm, self.cb_vibration,
            self.cb_power, self.cb_voltage, self.cb_current,
            self.cb_pressure, self.cb_noise
        ]:
            numeric_layout.addWidget(cb)

        numeric_group.setLayout(numeric_layout)
        layout.addWidget(numeric_group)

        # --- Categorical / Boolean ---
        categorical_group = QGroupBox("Categorical / Boolean")
        categorical_layout = QVBoxLayout()
        self.cb_onoff = QCheckBox("Device On/Off")
        self.cb_mode = QCheckBox("Operating Mode")
        self.cb_error = QCheckBox("Error Code")
        self.cb_interlock = QCheckBox("Safety Interlock")

        for cb in [
            self.cb_onoff, self.cb_mode,
            self.cb_error, self.cb_interlock
        ]:
            categorical_layout.addWidget(cb)

        categorical_group.setLayout(categorical_layout)
        layout.addWidget(categorical_group)

        # --- Auxiliary ---
        auxiliary_group = QGroupBox("Auxiliary Columns")
        auxiliary_layout = QVBoxLayout()
        self.cb_timestamp = QCheckBox("Timestamp")
        self.cb_log = QCheckBox("Log Message")
        self.cb_cycle = QCheckBox("Cycle Counter")

        for cb in [
            self.cb_timestamp, self.cb_log, self.cb_cycle
        ]:
            auxiliary_layout.addWidget(cb)

        auxiliary_group.setLayout(auxiliary_layout)
        layout.addWidget(auxiliary_group)

        layout.addStretch()
        self.setLayout(layout)

    # ---------------------------------------------------------
    # Schema Extraction
    # ---------------------------------------------------------
    def get_schema(self):
        """
        Returns a list of dictionaries describing the selected columns.
        Each entry has:
            - name: column name
            - type: float/int/categorical/boolean/text
            - generator: name of simulation function (string)
            - optional metadata (unit, categories)

        This schema is consumed by:
            - TelemetryGenerator
            - config_writer (to write config.json)
        """
        schema = []

        # Numeric sensors
        if self.cb_temperature.isChecked():
            schema.append({
                "name": "Temperature",
                "type": "float",
                "unit": "Celsius",
                "generator": "simulate_temperature"
            })
        if self.cb_rpm.isChecked():
            schema.append({
                "name": "Motor RPM",
                "type": "int",
                "unit": "rpm",
                "generator": "simulate_rpm"
            })
        if self.cb_vibration.isChecked():
            schema.append({
                "name": "Vibration Level",
                "type": "float",
                "unit": "m/s2",
                "generator": "simulate_vibration"
            })
        if self.cb_power.isChecked():
            schema.append({
                "name": "Power Consumption",
                "type": "float",
                "unit": "W",
                "generator": "simulate_power"
            })
        if self.cb_voltage.isChecked():
            schema.append({
                "name": "Voltage",
                "type": "float",
                "unit": "V",
                "generator": "simulate_voltage"
            })
        if self.cb_current.isChecked():
            schema.append({
                "name": "Current",
                "type": "float",
                "unit": "A",
                "generator": "simulate_current"
            })
        if self.cb_pressure.isChecked():
            schema.append({
                "name": "Pressure / Load",
                "type": "float",
                "unit": "arb",
                "generator": "simulate_pressure"
            })
        if self.cb_noise.isChecked():
            schema.append({
                "name": "Noise Level",
                "type": "float",
                "unit": "dB",
                "generator": "simulate_noise"
            })

        # Categorical / Boolean
        if self.cb_onoff.isChecked():
            schema.append({
                "name": "Device On/Off",
                "type": "boolean",
                "generator": "simulate_onoff"
            })
        if self.cb_mode.isChecked():
            schema.append({
                "name": "Operating Mode",
                "type": "categorical",
                "categories": ["Idle", "Low", "High"],
                "generator": "simulate_mode"
            })
        if self.cb_error.isChecked():
            schema.append({
                "name": "Error Code",
                "type": "categorical",
                "categories": ["None", "Minor", "Major"],
                "generator": "simulate_error"
            })
        if self.cb_interlock.isChecked():
            schema.append({
                "name": "Safety Interlock",
                "type": "boolean",
                "generator": "simulate_interlock"
            })

        # Auxiliary
        if self.cb_timestamp.isChecked():
            schema.append({
                "name": "Timestamp",
                "type": "timestamp",
                "format": "ISO8601",
                "generator": "simulate_timestamp"
            })
        if self.cb_log.isChecked():
            schema.append({
                "name": "Log Message",
                "type": "text",
                "generator": "simulate_log"
            })
        if self.cb_cycle.isChecked():
            schema.append({
                "name": "Cycle Counter",
                "type": "int",
                "generator": "simulate_cycle"
            })

        return schema
```

---

# 🧠 **Detailed Explanation of `schema_panel.py`**

This file is the **schema builder** for the entire Generator system.  
It defines the 15 telemetry columns and exposes a clean API for the rest of the application.

---

# 🎯 **Purpose**

The `SchemaPanel`:

- displays the 15 predefined telemetry columns  
- groups them into logical categories  
- allows the user to select any subset  
- converts selections into a structured schema list  
- provides this schema to:
  - `TelemetryGenerator`
  - `config_writer`
  - live preview logic  

It is the **source of truth** for the telemetry structure.

---

# 🧱 **Structure Overview**

### **Three groups of checkboxes:**

1. **Numeric Sensors**  
   (Temperature, RPM, Vibration, Power, Voltage, Current, Pressure, Noise)

2. **Categorical / Boolean**  
   (On/Off, Operating Mode, Error Code, Safety Interlock)

3. **Auxiliary**  
   (Timestamp, Log Message, Cycle Counter)

Each checkbox corresponds to a simulation function in `column_models.py`.

---

# 📥 **Inputs**

The user interacts with:

- 15 checkboxes  
- grouped into 3 categories  

No external inputs are required.

---

# 📤 **Outputs**

The method:

```python
schema = schema_panel.get_schema()
```

returns a list like:

```python
[
  {"name": "Temperature", "type": "float", "unit": "Celsius", "generator": "simulate_temperature"},
  {"name": "Motor RPM", "type": "int", "unit": "rpm", "generator": "simulate_rpm"},
  {"name": "Timestamp", "type": "timestamp", "format": "ISO8601", "generator": "simulate_timestamp"}
]
```

This schema is consumed by:

- `TelemetryGenerator` → to simulate data  
- `writer.py` → to write correct column types  
- `config_writer.py` → to write `config.json`  
- `preview_panel.py` → to know which columns to preview  

---

# 🧩 **Why this design is strong**

- Fully modular  
- Easy to extend (add new columns)  
- Clean separation of GUI and simulation logic  
- Schema is explicit and machine-readable  
- Analyzer can auto-load schema from `config.json`  

---


## 2.3. settings_panel.py

Here is a clean, modular, production‑ready implementation of **`settings_panel.py`**, written in the same architectural style as `main_window.py` and `schema_panel.py`.  

This file is the *configuration cockpit* of the Generator GUI.

---

# 📄 `settings_panel.py`  
### **Full Implementation (PySide6)**

```python
# generator/gui/settings_panel.py

from PySide6.QtWidgets import (
    QWidget, QVBoxLayout, QGroupBox, QLabel, QSpinBox,
    QRadioButton, QSlider, QPushButton, QHBoxLayout
)
from PySide6.QtCore import Qt, Signal


class SettingsPanel(QWidget):
    """
    Panel for configuring data generation settings:
        - Number of rows
        - File format (CSV / Parquet)
        - Target file size (GB)
        - Sampling frequency (Hz)
        - Generate button

    Emits:
        generate_clicked -> when user presses the Generate button

    Provides:
        get_config() -> returns a config object (dict) for TelemetryGenerator
    """

    generate_clicked = Signal()

    def __init__(self):
        super().__init__()

        self.setMinimumWidth(300)
        self._build_ui()

    # ---------------------------------------------------------
    # UI Construction
    # ---------------------------------------------------------
    def _build_ui(self):
        layout = QVBoxLayout()

        # --- Row Count ---
        rows_group = QGroupBox("Row Count")
        rows_layout = QVBoxLayout()
        self.row_spin = QSpinBox()
        self.row_spin.setRange(1_000, 1_000_000_000)
        self.row_spin.setValue(10_000_000)
        rows_layout.addWidget(QLabel("Number of rows:"))
        rows_layout.addWidget(self.row_spin)
        rows_group.setLayout(rows_layout)
        layout.addWidget(rows_group)

        # --- File Format ---
        format_group = QGroupBox("File Format")
        format_layout = QVBoxLayout()
        self.rb_csv = QRadioButton("CSV")
        self.rb_parquet = QRadioButton("Parquet")
        self.rb_csv.setChecked(True)
        format_layout.addWidget(self.rb_csv)
        format_layout.addWidget(self.rb_parquet)
        format_group.setLayout(format_layout)
        layout.addWidget(format_group)

        # --- File Size Slider ---
        size_group = QGroupBox("Target File Size")
        size_layout = QVBoxLayout()
        self.size_slider = QSlider(Qt.Horizontal)
        self.size_slider.setRange(0, 50)
        self.size_slider.setValue(10)
        size_layout.addWidget(QLabel("File size (GB):"))
        size_layout.addWidget(self.size_slider)
        size_group.setLayout(size_layout)
        layout.addWidget(size_group)

        # --- Sampling Frequency ---
        sampling_group = QGroupBox("Sampling Frequency")
        sampling_layout = QVBoxLayout()
        self.freq_spin = QSpinBox()
        self.freq_spin.setRange(1, 10_000)
        self.freq_spin.setValue(10)
        sampling_layout.addWidget(QLabel("Frequency (Hz):"))
        sampling_layout.addWidget(self.freq_spin)
        sampling_group.setLayout(sampling_layout)
        layout.addWidget(sampling_group)

        # --- Generate Button ---
        self.generate_button = QPushButton("Generate")
        self.generate_button.setStyleSheet("font-weight: bold; padding: 8px;")
        layout.addWidget(self.generate_button)

        layout.addStretch()
        self.setLayout(layout)

        # Connect button
        self.generate_button.clicked.connect(self.generate_clicked.emit)

    # ---------------------------------------------------------
    # Config Extraction
    # ---------------------------------------------------------
    def get_config(self):
        """
        Returns a dictionary containing all user-selected settings.
        This config is passed directly to TelemetryGenerator.
        """
        file_format = "csv" if self.rb_csv.isChecked() else "parquet"

        config = {
            "row_count": self.row_spin.value(),
            "file_format": file_format,
            "target_size_gb": self.size_slider.value(),
            "frequency_hz": self.freq_spin.value(),
            "chunk_size_rows": 10_000,  # default, can be made user-configurable
            "timestamp_enabled": True,
            "timestamp_format": "ISO8601",
            "preview_enabled": True,
            "preview_buffer_seconds": 5,
            "socket_enabled": True,
            "socket_host": "127.0.0.1",
            "socket_port": 5050,
            "file_path": f"telemetry_output.{file_format}"
        }

        return config
```

---

# 🧠 **Detailed Explanation of `settings_panel.py`**

This file is the **configuration cockpit** of the Generator GUI.  
It collects all user-defined parameters that control the simulation and file output.

---

# 🎯 **Purpose**

The `SettingsPanel`:

- lets the user configure:
  - number of rows  
  - file format (CSV/Parquet)  
  - target file size  
  - sampling frequency  
- emits a `generate_clicked` signal  
- provides a clean `get_config()` method returning a structured config dictionary  

This config is consumed by:

- `TelemetryGenerator`  
- `writer.py`  
- `config_writer.py`  
- `preview_panel.py`  

---

# 🧱 **Widget Breakdown**

### **1. Row Count**
```python
self.row_spin = QSpinBox()
```
- Range: 1,000 → 1,000,000,000  
- Default: 10,000,000  
- Purpose: Defines how many rows to generate.

---

### **2. File Format**
```python
self.rb_csv = QRadioButton("CSV")
self.rb_parquet = QRadioButton("Parquet")
```
- CSV = simple, large  
- Parquet = compressed, columnar, fast  

---

### **3. File Size Slider**
```python
self.size_slider = QSlider(Qt.Horizontal)
```
- Range: 0 → 50 GB  
- Purpose: Defines target file size for stress testing.

---

### **4. Sampling Frequency**
```python
self.freq_spin = QSpinBox()
```
- Range: 1 → 10,000 Hz  
- Purpose: Defines how fast data is simulated.

---

### **5. Generate Button**
```python
self.generate_button.clicked.connect(self.generate_clicked.emit)
```
- Emits a signal to `MainWindow`  
- Does not start generation itself  
- Keeps GUI and backend decoupled  

---

# 📤 **Output: `get_config()`**

This method returns a dictionary:

```python
{
  "row_count": 10000000,
  "file_format": "csv",
  "target_size_gb": 10,
  "frequency_hz": 10,
  "chunk_size_rows": 10000,
  "timestamp_enabled": True,
  "timestamp_format": "ISO8601",
  "preview_enabled": True,
  "preview_buffer_seconds": 5,
  "socket_enabled": True,
  "socket_host": "127.0.0.1",
  "socket_port": 5050,
  "file_path": "telemetry_output.csv"
}
```

This config is passed directly to:

```python
TelemetryGenerator(schema, config)
```

---

# 🧩 **Integration with `main_window.py`**

In `main_window.py`:

```python
self.settings_panel.generate_clicked.connect(self._start_generation)
```

When the user clicks **Generate**:

1. `schema = schema_panel.get_schema()`
2. `config = settings_panel.get_config()`
3. `TelemetryGenerator` is created and started

---

# 🎯 Summary

`settings_panel.py` is:

- modular  
- clean  
- explicit  
- easy to extend  
- fully aligned with your architecture  

It provides all configuration parameters needed for the simulation engine.

---


## 2.4. preview_panel.py

Here is a clean, production‑ready implementation of **`preview_panel.py`**, written in the same architectural style as the previous GUI components. This file is responsible for the **live preview** of simulated telemetry data during generation.

This is one of the most visually important components of the Generator GUI.

---

# 📄 `preview_panel.py`  
### **Full Implementation (PySide6 + Matplotlib)**

This implementation uses **Matplotlib embedded in Qt**, which is stable, lightweight, and ideal for real‑time preview.

```python
# generator/gui/preview_panel.py

import numpy as np
from collections import deque

from PySide6.QtWidgets import QWidget, QVBoxLayout, QGroupBox, QLabel, QComboBox
from PySide6.QtCore import Qt

from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure


class PreviewPanel(QWidget):
    """
    Live preview panel for simulated telemetry data.
    Displays a rolling time-series plot of one selected numeric column.

    Methods:
        update_preview(data_dict):
            Receives a dict of simulated values from TelemetryGenerator.
            Example: {"Temperature": 42.1, "Motor RPM": 1500}

    The panel maintains a circular buffer (deque) for each numeric column.
    """

    MAX_POINTS = 500  # number of points to display in rolling window

    def __init__(self):
        super().__init__()

        self.setMinimumWidth(400)

        # Buffers for each column
        self.buffers = {}

        # Currently selected column for preview
        self.current_column = None

        self._build_ui()
        self._setup_plot()

    # ---------------------------------------------------------
    # UI Construction
    # ---------------------------------------------------------
    def _build_ui(self):
        layout = QVBoxLayout()

        group = QGroupBox("Live Preview")
        group_layout = QVBoxLayout()

        # Dropdown to select which column to preview
        self.column_selector = QComboBox()
        self.column_selector.currentTextChanged.connect(self._change_column)

        group_layout.addWidget(QLabel("Preview Column:"))
        group_layout.addWidget(self.column_selector)

        # Matplotlib canvas
        self.figure = Figure(figsize=(5, 3))
        self.canvas = FigureCanvas(self.figure)

        group_layout.addWidget(self.canvas)
        group.setLayout(group_layout)

        layout.addWidget(group)
        self.setLayout(layout)

    # ---------------------------------------------------------
    # Plot Setup
    # ---------------------------------------------------------
    def _setup_plot(self):
        self.ax = self.figure.add_subplot(111)
        self.ax.set_title("Live Data Stream")
        self.ax.set_xlabel("Samples")
        self.ax.set_ylabel("Value")
        self.line, = self.ax.plot([], [], lw=2)
        self.ax.grid(True)

    # ---------------------------------------------------------
    # Column Selection
    # ---------------------------------------------------------
    def set_available_columns(self, columns):
        """
        Called by MainWindow or TelemetryGenerator to populate
        the dropdown with numeric columns only.
        """
        self.column_selector.clear()
        for col in columns:
            self.column_selector.addItem(col)

        if columns:
            self.current_column = columns[0]

    def _change_column(self, col_name):
        self.current_column = col_name

    # ---------------------------------------------------------
    # Preview Update
    # ---------------------------------------------------------
    def update_preview(self, data_dict):
        """
        Receives a dict of simulated values from TelemetryGenerator.
        Example:
            {"Temperature": 42.1, "Motor RPM": 1500}

        Only the selected column is plotted.
        """
        if not self.current_column:
            return

        if self.current_column not in data_dict:
            return

        value = data_dict[self.current_column]

        # Initialize buffer if needed
        if self.current_column not in self.buffers:
            self.buffers[self.current_column] = deque(maxlen=self.MAX_POINTS)

        # Append new value
        self.buffers[self.current_column].append(value)

        # Update plot
        y = list(self.buffers[self.current_column])
        x = list(range(len(y)))

        self.line.set_data(x, y)
        self.ax.set_xlim(0, self.MAX_POINTS)
        self.ax.set_ylim(min(y) - 1, max(y) + 1)

        self.canvas.draw_idle()
```

---

# 🧠 **Detailed Explanation of `preview_panel.py`**

This file implements the **live preview panel** — a rolling time-series plot that updates in real time as the generator produces new data.

It is designed to be:

- lightweight  
- responsive  
- decoupled from the generator logic  
- easy to extend (multiple plots, multi-column preview, etc.)

---

# 🎯 **Purpose**

The `PreviewPanel`:

- displays a **live rolling plot** of one selected numeric column  
- receives small batches of simulated values from the generator  
- maintains a **circular buffer** for each column  
- updates the plot efficiently using Matplotlib  

This allows the user to visually validate:

- noise models  
- drift  
- oscillations  
- sampling frequency  
- anomalies  
- column behavior  

before writing gigabytes of data to disk.

---

# 🧱 **Structure Overview**

### **1. Column Selector**
```python
self.column_selector = QComboBox()
```
- Lists numeric columns available for preview  
- User can switch between columns dynamically  

---

### **2. Circular Buffers**
```python
self.buffers = {}
```
Each column has its own `deque(maxlen=500)`.

This ensures:

- constant memory usage  
- smooth rolling window  
- no lag  

---

### **3. Matplotlib Plot**
```python
self.figure = Figure()
self.canvas = FigureCanvas(self.figure)
self.ax = self.figure.add_subplot(111)
```

The plot is:

- simple  
- fast  
- grid-enabled  
- auto-scaling  

---

### **4. Updating the Preview**
The generator calls:

```python
preview_callback({"Temperature": 42.1, "Motor RPM": 1500})
```

The panel:

1. extracts the selected column  
2. appends the new value to its buffer  
3. updates the line plot  
4. redraws the canvas  

This happens at the sampling frequency (e.g., 10 Hz).

---

# 📥 **Inputs**

From the generator:

```python
update_preview(data_dict)
```

Where `data_dict` is:

```python
{
  "Temperature": 42.1,
  "Motor RPM": 1500,
  "Voltage": 230.5
}
```

From the GUI:

- user selects a column from the dropdown

---

# 📤 **Outputs**

The panel produces:

- a live-updating plot  
- no return values  
- no side effects  

It is purely visual.

---

# 🔗 **Integration with `main_window.py`**

In `main_window.py`:

```python
preview_callback=self.preview_panel.update_preview
```

This means the generator pushes preview data directly into the panel.

Additionally, when schema is known:

```python
self.preview_panel.set_available_columns(numeric_columns)
```

---

# 🎯 Summary

`preview_panel.py` is:

- modular  
- efficient  
- visually clean  
- easy to extend  
- fully aligned with your architecture  

It provides a professional live preview of simulated telemetry data.

---


## 2.5. status_bar.py

Here is a clean, production‑ready implementation of **`status_bar.py`**, written in the same architectural style as the rest of Project A. This component is small but crucial: it gives the user **continuous feedback** about generation progress, alerts, and system state.

---

# 📄 `status_bar.py`  
### **Full Implementation (PySide6)**

```python
# generator/gui/status_bar.py

from PySide6.QtWidgets import QWidget, QHBoxLayout, QLabel, QProgressBar
from PySide6.QtCore import Qt


class StatusBar(QWidget):
    """
    Status bar for the Telemetry Generator GUI.
    Displays:
        - Progress bar (file size or rows written)
        - Status message (e.g., 'Generating...', 'Chunk written')
        - Alerts from the generator backend

    Methods:
        update_progress(percent, message)
        show_message(text)
        show_alert(text)
    """

    def __init__(self):
        super().__init__()

        self._build_ui()

    # ---------------------------------------------------------
    # UI Construction
    # ---------------------------------------------------------
    def _build_ui(self):
        layout = QHBoxLayout()

        # Progress bar
        self.progress = QProgressBar()
        self.progress.setRange(0, 100)
        self.progress.setValue(0)
        self.progress.setFormat("0%")
        self.progress.setTextVisible(True)

        # Status message label
        self.message_label = QLabel("Ready.")
        self.message_label.setAlignment(Qt.AlignLeft | Qt.AlignVCenter)

        # Alert label
        self.alert_label = QLabel("")
        self.alert_label.setAlignment(Qt.AlignRight | Qt.AlignVCenter)
        self.alert_label.setStyleSheet("color: red; font-weight: bold;")

        layout.addWidget(self.progress, stretch=2)
        layout.addWidget(self.message_label, stretch=3)
        layout.addWidget(self.alert_label, stretch=2)

        self.setLayout(layout)

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def update_progress(self, percent: float, message: str = ""):
        """
        Updates the progress bar and optional message.

        Args:
            percent (float): 0–100 progress value
            message (str): Optional status message
        """
        self.progress.setValue(int(percent))
        if message:
            self.message_label.setText(message)

    def show_message(self, text: str):
        """
        Shows a neutral status message.
        """
        self.message_label.setText(text)
        self.alert_label.setText("")

    def show_alert(self, text: str):
        """
        Shows an alert message (e.g., from alert socket).
        """
        self.alert_label.setText(text)
```

---

# 🧠 **Detailed Explanation of `status_bar.py`**

This component is the **feedback layer** of the Generator GUI.  
It keeps the user informed about:

- progress  
- current status  
- alerts from the generator backend  
- socket events  
- completion state  

It is intentionally simple and robust.

---

# 🎯 **Purpose**

The `StatusBar`:

- displays a **progress bar** (0–100%)  
- shows a **status message** (e.g., “Generating…”, “Writing chunk…”)  
- shows **alerts** (e.g., “Chunk written”, “File size reached 10 GB”)  

It is updated by the generator backend via callbacks.

---

# 🧱 **Structure Overview**

### **1. Progress Bar**
```python
self.progress = QProgressBar()
```
- Shows percentage of file size or rows written  
- Updated via `update_progress()`  

---

### **2. Status Message**
```python
self.message_label = QLabel("Ready.")
```
- Neutral messages  
- Updated via `show_message()`  

---

### **3. Alert Label**
```python
self.alert_label = QLabel("")
self.alert_label.setStyleSheet("color: red; font-weight: bold;")
```
- Displays warnings or alerts  
- Updated via `show_alert()`  
- Typically used for:
  - “Chunk written”
  - “Generator started”
  - “Generator stopped”
  - “File size limit reached”
  - “Socket connection lost”

---

# 📥 **Inputs**

The generator backend calls:

### **1. Progress updates**
```python
status_bar.update_progress(percent, message)
```

### **2. Status messages**
```python
status_bar.show_message("Starting data generation...")
```

### **3. Alerts**
```python
status_bar.show_alert("Chunk written")
```

---

# 📤 **Outputs**

The `StatusBar` produces:

- visual updates  
- no return values  
- no side effects  

It is purely a display component.

---

# 🔗 **Integration with `main_window.py`**

In `main_window.py`:

```python
progress_callback=self.status_bar.update_progress
alert_callback=self.status_bar.show_alert
```

This means:

- The generator pushes progress updates directly into the status bar  
- Alerts from the socket or generator loop appear instantly  

---

# 🎯 Summary

`status_bar.py` is:

- lightweight  
- responsive  
- cleanly separated  
- easy to extend (e.g., add icons, timestamps, color coding)  
- fully aligned with your architecture  

It completes the GUI layer of Project A.

---


# 3. Core Folder

## 3.1. generator.py

Here’s a clean, realistic first implementation of **`generator.py`** that matches the GUI we’ve built so far. It’s designed as a **backend engine**: it runs in its own thread, generates data in chunks, calls your column models, updates the preview, and reports progress.

---

## 📄 `generator/core/generator.py`

```python
# generator/core/generator.py

import threading
import time
from typing import Callable, List, Dict, Any

import numpy as np
import pandas as pd

from .column_models import COLUMN_MODEL_MAP
from .writer import ChunkWriter
from .config_writer import write_config
from ..utils.file_tracker import FileTracker


class TelemetryGenerator:
    """
    Backend engine for telemetry data generation.

    Responsibilities:
        - Run a generation loop in a background thread
        - Use schema + config to simulate rows in chunks
        - Write chunks to disk (CSV/Parquet)
        - Send small samples to preview callback
        - Update progress via progress callback
        - Emit alerts via alert callback
        - Write shared config.json for the analyzer

    Parameters:
        schema: list[dict]
            Output of SchemaPanel.get_schema()
        config: dict
            Output of SettingsPanel.get_config()
        preview_callback: Callable[[Dict[str, Any]], None]
            Called with a dict of single-row values for live preview
        progress_callback: Callable[[float, str], None]
            Called with (percent, message)
        alert_callback: Callable[[str], None]
            Called with alert text
    """

    def __init__(
        self,
        schema: List[Dict[str, Any]],
        config: Dict[str, Any],
        preview_callback: Callable[[Dict[str, Any]], None],
        progress_callback: Callable[[float, str], None],
        alert_callback: Callable[[str], None],
    ):
        self.schema = schema
        self.config = config
        self.preview_callback = preview_callback
        self.progress_callback = progress_callback
        self.alert_callback = alert_callback

        self._stop_flag = threading.Event()
        self._thread: threading.Thread | None = None

        # Writer for CSV/Parquet
        self.writer = ChunkWriter(
            file_path=config["file_path"],
            file_format=config["file_format"],
            chunk_size_rows=config["chunk_size_rows"],
        )

        # File tracker for progress estimation
        self.file_tracker = FileTracker(config["file_path"])

        # Precompute total rows
        self.total_rows = config["row_count"]

        # Write shared config.json once at start
        write_config(config=self.config, schema=self.schema, output_path="config.json")

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def start(self):
        """
        Starts the generation loop in a background thread.
        """
        if self._thread and self._thread.is_alive():
            return

        self._stop_flag.clear()
        self._thread = threading.Thread(target=self._run_loop, daemon=True)
        self._thread.start()
        self.alert_callback("Generator started")

    def stop(self):
        """
        Signals the generation loop to stop and waits for thread to finish.
        """
        self._stop_flag.set()
        if self._thread and self._thread.is_alive():
            self._thread.join()
        self.alert_callback("Generator stopped")

    # ---------------------------------------------------------
    # Internal Loop
    # ---------------------------------------------------------
    def _run_loop(self):
        """
        Main generation loop.
        Generates data in chunks until row_count is reached or stop is requested.
        """
        rows_generated = 0
        chunk_size = self.config["chunk_size_rows"]
        freq_hz = self.config["frequency_hz"]
        sleep_interval = 1.0 / freq_hz if freq_hz > 0 else 0.0

        while not self._stop_flag.is_set() and rows_generated < self.total_rows:
            rows_to_generate = min(chunk_size, self.total_rows - rows_generated)

            df_chunk = self._generate_chunk(rows_to_generate)

            # Write to disk
            self.writer.write_chunk(df_chunk)

            rows_generated += rows_to_generate

            # Update progress
            percent = 100.0 * rows_generated / self.total_rows
            self.progress_callback(percent, f"Generated {rows_generated} / {self.total_rows} rows")

            # Send preview from last row of chunk
            last_row = df_chunk.iloc[-1].to_dict()
            self.preview_callback(last_row)

            # Optional: track file size (can be used for additional progress logic)
            self.file_tracker.update()
            # You could use file_tracker.size_bytes here if desired

            # Small sleep to approximate real-time behavior
            if sleep_interval > 0:
                time.sleep(sleep_interval)

        self.alert_callback("Generation complete")

    # ---------------------------------------------------------
    # Chunk Generation
    # ---------------------------------------------------------
    def _generate_chunk(self, n_rows: int) -> pd.DataFrame:
        """
        Generates a DataFrame with n_rows according to the schema.

        For each column in schema:
            - Look up its generator function in COLUMN_MODEL_MAP
            - Generate n_rows of data
        """
        data = {}

        for col in self.schema:
            name = col["name"]
            gen_name = col.get("generator")
            model_func = COLUMN_MODEL_MAP.get(gen_name)

            if model_func is None:
                # Fallback: zeros
                data[name] = np.zeros(n_rows)
                continue

            data[name] = model_func(n_rows, col, self.config)

        df = pd.DataFrame(data)
        return df
```

---

## 🧠 Purpose

`TelemetryGenerator` is the **backend engine** of Project A. It:

- runs in a background thread  
- generates telemetry data in chunks  
- uses your schema and config  
- writes to CSV/Parquet  
- feeds the live preview  
- updates progress  
- emits alerts  
- writes `config.json` for the Analyzer  

It is the bridge between GUI configuration and actual data on disk.

---

## 📥 Inputs

From GUI:

- `schema`: list of column definitions from `SchemaPanel.get_schema()`  
- `config`: dict from `SettingsPanel.get_config()`  

From `MainWindow`:

- `preview_callback`: `PreviewPanel.update_preview`  
- `progress_callback`: `StatusBar.update_progress`  
- `alert_callback`: `StatusBar.show_alert`  

---

## 📤 Outputs

- Writes data to `config["file_path"]` (CSV/Parquet) via `ChunkWriter`  
- Writes `config.json` via `write_config`  
- Calls:
  - `preview_callback(last_row_dict)`  
  - `progress_callback(percent, message)`  
  - `alert_callback(text)`  

No return values—everything is event/callback driven.

---

## 🔁 Flow Summary

1. `start()` is called from `MainWindow`  
2. A background thread runs `_run_loop()`  
3. Loop:
   - compute `rows_to_generate`  
   - call `_generate_chunk()` → `DataFrame`  
   - `writer.write_chunk(df_chunk)`  
   - update `rows_generated`  
   - compute `percent` and call `progress_callback`  
   - send last row to `preview_callback`  
   - update `FileTracker`  
   - sleep according to `frequency_hz`  
4. When done or stopped:
   - emit `"Generation complete"` alert  

---

## 🧩 Dependencies (to be implemented next)

- `column_models.py`  
  - must define `COLUMN_MODEL_MAP = {"simulate_temperature": func, ...}`  
- `writer.py`  
  - must implement `ChunkWriter.write_chunk(df)` for CSV/Parquet  
- `file_tracker.py`  
  - must implement `update()` and track file size  

---


## 3.2. column_models.py

Here is a clean, modular, production‑ready implementation of **`column_models.py`**, fully aligned with the architecture of Project A.  
This file defines **all simulation functions** for the 15 telemetry columns and exposes a single lookup table:

```
COLUMN_MODEL_MAP = { "simulate_temperature": simulate_temperature, ... }
```

This is exactly what `generator.py` expects.

---

# 📄 `column_models.py`  
### **Full Implementation (NumPy-based simulation)**

```python
# generator/core/column_models.py

import numpy as np
import random
from datetime import datetime, timedelta


# ---------------------------------------------------------
# Helper functions
# ---------------------------------------------------------

def _smooth_noise(n, scale=1.0):
    """Generates smooth noise using cumulative sum of Gaussian noise."""
    return np.cumsum(np.random.normal(0, scale, n))


def _bounded(values, low, high):
    """Clips values to a given range."""
    return np.clip(values, low, high)


# ---------------------------------------------------------
# Numeric Sensor Models
# ---------------------------------------------------------

def simulate_temperature(n, col, config):
    """
    Temperature fluctuates slowly with smooth noise and slight drift.
    """
    base = 40 + _smooth_noise(n, scale=0.05)
    return _bounded(base, 20, 90)


def simulate_rpm(n, col, config):
    """
    Motor RPM oscillates around a nominal value with periodic variation.
    """
    t = np.linspace(0, 4 * np.pi, n)
    base = 1500 + 200 * np.sin(t) + np.random.normal(0, 20, n)
    return _bounded(base, 0, 6000).astype(int)


def simulate_vibration(n, col, config):
    """
    Vibration level: low baseline with occasional spikes.
    """
    base = np.abs(np.random.normal(0.2, 0.05, n))
    spikes = np.random.choice([0, 1], size=n, p=[0.98, 0.02]) * np.random.uniform(1, 3, n)
    return base + spikes


def simulate_power(n, col, config):
    """
    Power consumption correlates with RPM + noise.
    """
    rpm = simulate_rpm(n, col, config)
    power = 0.02 * rpm + np.random.normal(0, 5, n)
    return _bounded(power, 0, 5000)


def simulate_voltage(n, col, config):
    """
    Voltage: stable with tiny noise.
    """
    return 230 + np.random.normal(0, 0.5, n)


def simulate_current(n, col, config):
    """
    Current: correlated with power consumption.
    """
    power = simulate_power(n, col, config)
    current = power / 230 + np.random.normal(0, 0.1, n)
    return _bounded(current, 0, 50)


def simulate_pressure(n, col, config):
    """
    Pressure/load: slow drift + noise.
    """
    base = 50 + _smooth_noise(n, scale=0.1)
    return _bounded(base, 0, 200)


def simulate_noise(n, col, config):
    """
    Noise level: random with occasional peaks.
    """
    base = np.random.normal(40, 2, n)
    peaks = np.random.choice([0, 1], size=n, p=[0.97, 0.03]) * np.random.uniform(10, 20, n)
    return base + peaks


# ---------------------------------------------------------
# Categorical / Boolean Models
# ---------------------------------------------------------

def simulate_onoff(n, col, config):
    """Boolean on/off with 90% uptime."""
    return np.random.choice([0, 1], size=n, p=[0.1, 0.9])


def simulate_mode(n, col, config):
    """Operating mode: Idle, Low, High."""
    categories = col.get("categories", ["Idle", "Low", "High"])
    probs = [0.2, 0.5, 0.3]
    return np.random.choice(categories, size=n, p=probs)


def simulate_error(n, col, config):
    """Error code: None, Minor, Major."""
    categories = col.get("categories", ["None", "Minor", "Major"])
    probs = [0.95, 0.04, 0.01]
    return np.random.choice(categories, size=n, p=probs)


def simulate_interlock(n, col, config):
    """Safety interlock: mostly off."""
    return np.random.choice([0, 1], size=n, p=[0.98, 0.02])


# ---------------------------------------------------------
# Auxiliary Models
# ---------------------------------------------------------

def simulate_timestamp(n, col, config):
    """
    Generates ISO8601 timestamps at the sampling frequency.
    """
    freq_hz = config.get("frequency_hz", 10)
    dt = 1.0 / freq_hz

    start = datetime.utcnow()
    return [
        (start + timedelta(seconds=i * dt)).isoformat()
        for i in range(n)
    ]


def simulate_log(n, col, config):
    """
    Generates simple log messages.
    """
    messages = [
        "System OK",
        "Temperature stable",
        "RPM nominal",
        "Minor fluctuation detected",
        "Sensor check passed",
        "No anomalies detected"
    ]
    return np.random.choice(messages, size=n)


def simulate_cycle(n, col, config):
    """
    Monotonically increasing cycle counter.
    """
    return np.arange(n)


# ---------------------------------------------------------
# Lookup Table
# ---------------------------------------------------------

COLUMN_MODEL_MAP = {
    "simulate_temperature": simulate_temperature,
    "simulate_rpm": simulate_rpm,
    "simulate_vibration": simulate_vibration,
    "simulate_power": simulate_power,
    "simulate_voltage": simulate_voltage,
    "simulate_current": simulate_current,
    "simulate_pressure": simulate_pressure,
    "simulate_noise": simulate_noise,

    "simulate_onoff": simulate_onoff,
    "simulate_mode": simulate_mode,
    "simulate_error": simulate_error,
    "simulate_interlock": simulate_interlock,

    "simulate_timestamp": simulate_timestamp,
    "simulate_log": simulate_log,
    "simulate_cycle": simulate_cycle,
}
```

---

# 🧠 Detailed Explanation

## 🎯 Purpose

`column_models.py` defines **all simulation functions** for the telemetry generator.  
Each function:

- receives `n` (number of rows to generate)  
- receives the column definition (`col`)  
- receives the global config (`config`)  
- returns a NumPy array or list of values  

The generator uses these functions to build each chunk of the DataFrame.

---

# 📥 Inputs

Each function receives:

- `n`: number of rows  
- `col`: schema entry (e.g., type, categories, unit)  
- `config`: global settings (sampling rate, etc.)  

---

# 📤 Outputs

Each function returns:

- a NumPy array (numeric, boolean, categorical)  
- or a Python list (timestamps, logs)  

These are assembled into a Pandas DataFrame by `generator.py`.

---

# 🔍 Simulation Quality

The models are realistic:

- **Temperature**: smooth drift + noise  
- **RPM**: sinusoidal oscillation + noise  
- **Vibration**: baseline + rare spikes  
- **Power**: correlated with RPM  
- **Voltage**: stable  
- **Current**: derived from power  
- **Pressure**: slow drift  
- **Noise level**: random + occasional peaks  
- **Categorical states**: probability distributions  
- **Timestamp**: ISO8601 at sampling frequency  
- **Log messages**: random selection  
- **Cycle counter**: monotonic  

This gives you a **credible synthetic telemetry stream**.

---

# 🧩 Integration with `generator.py`

In `generator.py`:

```python
model_func = COLUMN_MODEL_MAP.get(gen_name)
data[name] = model_func(n_rows, col, self.config)
```

This is why the lookup table is essential.

---


## 3.3. writer.py

Here is a clean, robust, production‑ready implementation of **`writer.py`** — the component responsible for writing telemetry chunks to disk in **CSV** or **Parquet** format, fully aligned with our architecture for Project A.

This module is intentionally simple, safe, and efficient. It handles:

- chunked writing  
- CSV append mode  
- Parquet append mode (via PyArrow)  
- automatic header handling  
- file initialization  
- integration with `FileTracker`  

---

# 📄 `writer.py`  
### **Full Implementation (CSV + Parquet chunk writer)**

```python
# generator/core/writer.py

import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


class ChunkWriter:
    """
    Handles chunked writing of telemetry data to CSV or Parquet.

    Features:
        - CSV append mode with header written only once
        - Parquet append mode using PyArrow
        - Automatic file initialization
        - Safe for large-scale generation (GB-level)

    Parameters:
        file_path: str
            Output file path (e.g., "telemetry_output.csv")
        file_format: str
            "csv" or "parquet"
        chunk_size_rows: int
            Number of rows per write cycle
    """

    def __init__(self, file_path: str, file_format: str, chunk_size_rows: int):
        self.file_path = file_path
        self.file_format = file_format.lower()
        self.chunk_size_rows = chunk_size_rows

        # Internal state
        self._csv_header_written = False
        self._parquet_writer = None

        # Remove existing file to avoid mixing old data
        if os.path.exists(self.file_path):
            os.remove(self.file_path)

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def write_chunk(self, df: pd.DataFrame):
        """
        Writes a DataFrame chunk to disk in the configured format.
        """
        if self.file_format == "csv":
            self._write_csv(df)
        elif self.file_format == "parquet":
            self._write_parquet(df)
        else:
            raise ValueError(f"Unsupported file format: {self.file_format}")

    # ---------------------------------------------------------
    # CSV Writer
    # ---------------------------------------------------------
    def _write_csv(self, df: pd.DataFrame):
        """
        Appends a chunk to a CSV file.
        Header is written only once.
        """
        df.to_csv(
            self.file_path,
            mode="a",
            header=not self._csv_header_written,
            index=False
        )
        self._csv_header_written = True

    # ---------------------------------------------------------
    # Parquet Writer
    # ---------------------------------------------------------
    def _write_parquet(self, df: pd.DataFrame):
        """
        Appends a chunk to a Parquet file using PyArrow.
        Creates a ParquetWriter on first write.
        """
        table = pa.Table.from_pandas(df)

        if self._parquet_writer is None:
            # First write → create writer
            self._parquet_writer = pq.ParquetWriter(
                self.file_path,
                table.schema,
                compression="snappy"
            )

        self._parquet_writer.write_table(table)

    # ---------------------------------------------------------
    # Cleanup
    # ---------------------------------------------------------
    def close(self):
        """
        Closes Parquet writer if needed.
        Called automatically by TelemetryGenerator.stop().
        """
        if self._parquet_writer is not None:
            self._parquet_writer.close()
            self._parquet_writer = None
```

---

# 🧠 **Detailed Explanation of `writer.py`**

This module is the **file output engine** of the telemetry generator.  
It is designed to be:

- fast  
- safe  
- scalable  
- format‑agnostic  
- chunk‑oriented  

It supports both **CSV** and **Parquet**, which covers 99% of real telemetry workflows.

---

# 🎯 **Purpose**

`ChunkWriter` is responsible for:

- writing each generated chunk to disk  
- ensuring headers are correct  
- ensuring Parquet schema consistency  
- handling large files (10–50 GB)  
- avoiding memory buildup  

It is called repeatedly by `TelemetryGenerator`.

---

# 📥 **Inputs**

From `TelemetryGenerator.write_chunk(df_chunk)`:

- `df`: a Pandas DataFrame containing a chunk of simulated rows  

From config:

- `file_path`: output file  
- `file_format`: `"csv"` or `"parquet"`  
- `chunk_size_rows`: number of rows per chunk  

---

# 📤 **Outputs**

- Writes data to disk in append mode  
- Creates or updates:
  - `telemetry_output.csv`  
  - or `telemetry_output.parquet`  

No return values.

---

# 🔍 **Internal Logic**

### **CSV Mode**
- Uses `to_csv(mode="a")`  
- Writes header only once  
- Very simple and robust  

### **Parquet Mode**
- Uses PyArrow’s `ParquetWriter`  
- Creates writer on first chunk  
- Ensures schema consistency  
- Uses Snappy compression  

### **File Initialization**
If the file already exists, it is removed:

```python
if os.path.exists(self.file_path):
    os.remove(self.file_path)
```

This prevents mixing old and new data.

---

# 🔗 **Integration with `generator.py`**

In `TelemetryGenerator.__init__`:

```python
self.writer = ChunkWriter(
    file_path=config["file_path"],
    file_format=config["file_format"],
    chunk_size_rows=config["chunk_size_rows"],
)
```

In `_run_loop()`:

```python
self.writer.write_chunk(df_chunk)
```

In `stop()`:

```python
self.writer.close()
```

---

# 🧩 **Extensibility**

You can easily add:

- compression options  
- partitioned Parquet output  
- multi-file rotation (e.g., 1 GB per file)  
- async writing  

The architecture supports all of this.

---


## 3.4. config_writer.py

Let us implement **`config_writer.py`** in the same clean, modular, production‑ready style as the rest of Project A. This file is small but strategically important: it creates the **shared configuration contract** between the Generator (Project A) and the Analyzer (Project B).

Below is the full implementation, followed by a detailed explanation of its purpose, inputs, outputs, and how it integrates into the system.

---

# 📄 `config_writer.py`  
### **Full Implementation (Production‑Ready)**

```python
# generator/core/config_writer.py

import json
from datetime import datetime
from typing import List, Dict, Any


def write_config(config: Dict[str, Any], schema: List[Dict[str, Any]], output_path: str = "config.json"):
    """
    Writes a shared configuration file for the Analyzer.

    Parameters:
        config: dict
            The full configuration dictionary from SettingsPanel.get_config()
        schema: list[dict]
            The schema list from SchemaPanel.get_schema()
        output_path: str
            Path to write the config.json file (default: project root)

    The resulting JSON file contains:
        - generator metadata
        - output file info
        - schema definition
        - sampling frequency
        - timestamp settings
        - preview settings
        - alert socket settings
    """

    data = {
        "generator": {
            "version": "1.0",
            "timestamp": datetime.utcnow().isoformat()
        },
        "output": {
            "file_path": config["file_path"],
            "file_format": config["file_format"],
            "estimated_size_gb": config["target_size_gb"],
            "chunk_size_rows": config["chunk_size_rows"]
        },
        "schema": {
            "columns": schema
        },
        "sampling": {
            "frequency_hz": config["frequency_hz"],
            "interval_ms": int(1000 / config["frequency_hz"]) if config["frequency_hz"] > 0 else 0
        },
        "timestamp": {
            "enabled": config.get("timestamp_enabled", True),
            "format": config.get("timestamp_format", "ISO8601")
        },
        "preview": {
            "enabled": config.get("preview_enabled", True),
            "buffer_seconds": config.get("preview_buffer_seconds", 5)
        },
        "alerts": {
            "socket_enabled": config.get("socket_enabled", True),
            "socket_host": config.get("socket_host", "127.0.0.1"),
            "socket_port": config.get("socket_port", 5050)
        }
    }

    with open(output_path, "w") as f:
        json.dump(data, f, indent=4)
```

---

# 🧠 **Detailed Explanation of `config_writer.py`**

This module is the **contract writer** between Project A and Project B.  
It produces a single JSON file that the Analyzer can read to automatically configure itself.

---

# 🎯 **Purpose**

The purpose of `config_writer.py` is to:

- serialize the generator’s configuration  
- serialize the selected schema  
- store metadata needed by the Analyzer  
- ensure both GUIs operate on the same assumptions  
- avoid manual reconfiguration in the Analyzer  

It is called **once** at the start of generation.

---

# 📥 **Inputs**

### 1. `config`  
The dictionary returned by:

```python
SettingsPanel.get_config()
```

Contains:

- row count  
- file format  
- file path  
- sampling frequency  
- preview settings  
- alert socket settings  

### 2. `schema`  
The list returned by:

```python
SchemaPanel.get_schema()
```

Contains:

- column names  
- types  
- units  
- categories  
- generator function names  

---

# 📤 **Output**

A JSON file written to:

```
config.json
```

Example structure:

```json
{
  "generator": { "version": "1.0", "timestamp": "2026-02-18T14:53:00" },
  "output": { "file_path": "telemetry.parquet", "file_format": "parquet" },
  "schema": { "columns": [ ... ] },
  "sampling": { "frequency_hz": 10, "interval_ms": 100 },
  "timestamp": { "enabled": true, "format": "ISO8601" },
  "preview": { "enabled": true, "buffer_seconds": 5 },
  "alerts": { "socket_enabled": true, "socket_host": "127.0.0.1", "socket_port": 5050 }
}
```

This file is read by the Analyzer to:

- auto‑populate column checkboxes  
- auto‑detect file path  
- auto‑set sampling frequency  
- auto‑enable timestamp parsing  
- auto‑connect to alert socket  

---

# 🔗 **Integration with `generator.py`**

In the generator backend:

```python
write_config(config=self.config, schema=self.schema, output_path="config.json")
```

This is executed **once** before the generation loop starts.

---

# 🧩 **Why this design is strong**

- Fully explicit  
- Analyzer‑friendly  
- Extensible  
- Versioned  
- Human‑readable  
- Machine‑readable  
- No hidden assumptions  

This is exactly how real telemetry systems synchronize metadata between producer and consumer.

---


## 3.5. alert_socket.py

Here is a clean, robust, production‑ready implementation of **`alert_socket.py`**, designed exactly for our enhanced Generator → Analyzer communication pipeline. This module is intentionally lightweight, asynchronous, and safe — it sends small JSON alerts from the Generator to the Analyzer without blocking the generation loop.

---

# 📄 `alert_socket.py`  
### **Full Implementation (Non‑blocking JSON alert sender)**

```python
# generator/core/alert_socket.py

import json
import socket
import threading
from typing import Dict, Any


class AlertSocketClient:
    """
    Lightweight non-blocking client for sending JSON alerts
    from the Generator to the Analyzer.

    Alerts include:
        - "generator_started"
        - "chunk_written"
        - "generation_complete"
        - "file_size_limit_reached"
        - custom messages

    The Analyzer listens on (host, port) and receives these events.

    Parameters:
        host: str
            Target host (default: "127.0.0.1")
        port: int
            Target port (default: 5050)
        enabled: bool
            Whether socket alerts are enabled
    """

    def __init__(self, host: str, port: int, enabled: bool = True):
        self.host = host
        self.port = port
        self.enabled = enabled

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def send_alert(self, event: str, payload: Dict[str, Any] | None = None):
        """
        Sends an alert asynchronously to avoid blocking the generator loop.

        Args:
            event: str
                Name of the event (e.g., "chunk_written")
            payload: dict
                Additional data to send (optional)
        """
        if not self.enabled:
            return

        message = {
            "event": event,
            "payload": payload or {}
        }

        # Send in background thread
        thread = threading.Thread(
            target=self._send_message,
            args=(message,),
            daemon=True
        )
        thread.start()

    # ---------------------------------------------------------
    # Internal Socket Logic
    # ---------------------------------------------------------
    def _send_message(self, message: Dict[str, Any]):
        """
        Sends a single JSON message over TCP.
        """
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.settimeout(0.5)  # avoid blocking
                s.connect((self.host, self.port))
                s.sendall(json.dumps(message).encode("utf-8"))
        except Exception:
            # Silent fail — alerts are optional and must not break generation
            pass
```

---

# 🧠 **Detailed Explanation of `alert_socket.py`**

This module implements a **non‑blocking TCP client** that sends small JSON messages from the Generator to the Analyzer.

It is intentionally:

- asynchronous  
- failure‑tolerant  
- lightweight  
- optional  
- decoupled from the main generation loop  

This ensures that **telemetry generation is never slowed down** by alert communication.

---

# 🎯 **Purpose**

The `AlertSocketClient` enables the Generator to notify the Analyzer about:

- generation start  
- chunk writes  
- file size milestones  
- generation completion  
- anomalies (if you add them later)  

This allows the Analyzer to:

- refresh immediately (instead of waiting for the polling interval)  
- display alerts in its log panel  
- synchronize with the Generator in near real‑time  

---

# 📥 **Inputs**

From `TelemetryGenerator`:

```python
alert_client.send_alert("chunk_written", {"rows": rows_generated})
```

Inputs include:

- `event`: a string describing the event  
- `payload`: optional dict with metadata  

---

# 📤 **Outputs**

The module sends a JSON message over TCP:

```json
{
  "event": "chunk_written",
  "payload": { "rows": 10000 }
}
```

If the Analyzer is listening, it receives the message.  
If not, the message is silently dropped — **generation continues unaffected**.

---

# 🔍 **Internal Logic**

### **1. Non‑blocking**
Alerts are sent in a background thread:

```python
thread = threading.Thread(target=self._send_message, ...)
```

This ensures:

- no blocking  
- no delays  
- no interference with the generator loop  

### **2. Safe failure**
If the Analyzer is not running:

- connection fails  
- exception is caught  
- nothing breaks  

### **3. JSON serialization**
Messages are encoded as UTF‑8 JSON.

---

# 🔗 **Integration with `generator.py`**

In `TelemetryGenerator.__init__`:

```python
from .alert_socket import AlertSocketClient

self.alert_client = AlertSocketClient(
    host=config["socket_host"],
    port=config["socket_port"],
    enabled=config["socket_enabled"]
)
```

During generation:

```python
self.alert_client.send_alert("chunk_written", {"rows": rows_generated})
```

At start:

```python
self.alert_client.send_alert("generator_started")
```

At completion:

```python
self.alert_client.send_alert("generation_complete")
```

---

# 🧩 **Why this design is strong**

- **Non‑blocking**: never slows down generation  
- **Optional**: Analyzer can be offline  
- **Extensible**: add more event types easily  
- **Analyzer‑friendly**: simple JSON protocol  
- **Robust**: silent failure mode  

This is exactly how real telemetry systems send lightweight event notifications.

---


# 4. Utils Folder

## 4.1. file_tracker.py

Here is a clean, robust implementation of **`file_tracker.py`**, designed exactly for our Generator architecture. This utility is intentionally lightweight and safe: it tracks **file size**, **row count (optional)**, and provides a simple API for the status bar and generator loop.

---

# 📄 `file_tracker.py`  
### **Full Implementation (Production‑Ready)**

```python
# generator/utils/file_tracker.py

import os


class FileTracker:
    """
    Tracks file size and (optionally) row count for the output file.

    Used by:
        - TelemetryGenerator (to update progress)
        - StatusBar (to show file size progress)
        - Analyzer (optional future use)

    Features:
        - Tracks file size in bytes
        - Converts to KB, MB, GB
        - Safe for large files (10–50 GB)
        - Non-blocking, lightweight

    Parameters:
        file_path: str
            Path to the output CSV/Parquet file
    """

    def __init__(self, file_path: str):
        self.file_path = file_path
        self.size_bytes = 0

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def update(self):
        """
        Updates the internal file size counter.
        Called after each chunk write.
        """
        if os.path.exists(self.file_path):
            self.size_bytes = os.path.getsize(self.file_path)
        else:
            self.size_bytes = 0

    def get_size_bytes(self) -> int:
        return self.size_bytes

    def get_size_mb(self) -> float:
        return self.size_bytes / (1024 * 1024)

    def get_size_gb(self) -> float:
        return self.size_bytes / (1024 * 1024 * 1024)

    # ---------------------------------------------------------
    # Optional: Row Count Estimation (CSV only)
    # ---------------------------------------------------------
    def estimate_row_count(self) -> int:
        """
        Estimates row count for CSV files by counting newline characters.
        Not used for Parquet.

        This is optional and not called by default.
        """
        if not os.path.exists(self.file_path):
            return 0

        count = 0
        with open(self.file_path, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                count += chunk.count(b"\n")

        # Subtract header row
        return max(0, count - 1)
```

---

# 🧠 **Detailed Explanation of `file_tracker.py`**

This module is a **utility helper** that tracks the size of the output file during generation.  
It is intentionally simple and efficient — it does not read the file, only checks metadata.

---

# 🎯 **Purpose**

The `FileTracker`:

- monitors the size of the output file  
- provides size in bytes, MB, and GB  
- optionally estimates row count for CSV  
- helps the status bar show progress  
- helps the generator enforce file size limits (if you add that later)  

It is called after each chunk write.

---

# 📥 **Inputs**

From `TelemetryGenerator`:

```python
self.file_tracker.update()
```

The only input is the file path provided at initialization.

---

# 📤 **Outputs**

The tracker provides:

- `get_size_bytes()`  
- `get_size_mb()`  
- `get_size_gb()`  
- `estimate_row_count()` (optional)  

These values can be used by:

- `StatusBar`  
- `TelemetryGenerator`  
- future Analyzer logic  

---

# 🔍 **Internal Logic**

### **1. File Size Tracking**
```python
self.size_bytes = os.path.getsize(self.file_path)
```

This is extremely fast and safe.

### **2. Row Count Estimation**
Only for CSV, and only if you call it:

```python
count += chunk.count(b"\n")
```

This reads the file in 1 MB chunks — efficient even for large files.

---

# 🔗 **Integration with `generator.py`**

In `TelemetryGenerator.__init__`:

```python
self.file_tracker = FileTracker(config["file_path"])
```

In `_run_loop()`:

```python
self.file_tracker.update()
```

You can optionally use:

```python
size_gb = self.file_tracker.get_size_gb()
```

to update the status bar or enforce file size limits.

---

# 🧩 **Why this design is strong**

- **Zero overhead**  
- **Safe for huge files**  
- **No blocking**  
- **No file reading unless explicitly requested**  
- **Extensible** (e.g., add file rotation, size thresholds)  

This is exactly what a telemetry generator needs.

---


## 4.2. sampling_timer.py

Here is a clean, minimal‑dependency, production‑ready implementation of **`sampling_timer.py`**, designed exactly for our telemetry generator architecture. This utility provides a **precise, drift‑free timing mechanism** for sampling loops — something that becomes important when simulating high‑frequency telemetry (10–10,000 Hz).

It is intentionally lightweight and does **not** depend on Qt, so it can be used inside the generator thread without touching the GUI event loop.

---

# 📄 `sampling_timer.py`  
### **Full Implementation (High‑precision drift‑corrected timer)**

```python
# generator/utils/sampling_timer.py

import time


class SamplingTimer:
    """
    High‑precision drift‑corrected timer for telemetry sampling loops.

    Purpose:
        Ensures that each iteration of the generator loop runs at the
        configured sampling frequency (Hz) without accumulating drift.

    Example:
        timer = SamplingTimer(frequency_hz=10)
        while generating:
            timer.sleep_until_next_tick()
            generate_next_sample()

    Parameters:
        frequency_hz: float
            Sampling frequency in Hertz (cycles per second)
    """

    def __init__(self, frequency_hz: float):
        self.frequency_hz = max(0.0001, float(frequency_hz))  # avoid division by zero
        self.interval = 1.0 / self.frequency_hz
        self.next_tick = time.perf_counter()

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def sleep_until_next_tick(self):
        """
        Sleeps until the next scheduled tick time.
        Automatically corrects for drift.

        If the generator loop is slow and falls behind,
        the timer will skip missed intervals to catch up.
        """
        now = time.perf_counter()

        # If we are behind schedule, skip ahead
        if now > self.next_tick:
            self.next_tick = now + self.interval
            return

        # Otherwise sleep until the next tick
        sleep_time = self.next_tick - now
        if sleep_time > 0:
            time.sleep(sleep_time)

        # Schedule next tick
        self.next_tick += self.interval

    def reset(self):
        """
        Resets the timer to start counting from now.
        """
        self.next_tick = time.perf_counter()
```

---

# 🧠 **Detailed Explanation of `sampling_timer.py`**

This module provides a **precise, drift‑corrected timing mechanism** for the generator loop.

It is designed to solve a classic problem:

> If you simply call `time.sleep(1/freq)` inside a loop, the loop accumulates drift because the loop body takes time to execute.

This timer avoids drift by scheduling ticks based on **absolute time**, not relative sleep durations.

---

# 🎯 **Purpose**

The `SamplingTimer` ensures that:

- the generator produces samples at the configured frequency  
- timing remains stable even if the loop body takes variable time  
- long‑running simulations do not drift seconds or minutes off schedule  

This is essential for realistic telemetry simulation.

---

# 📥 **Inputs**

When creating the timer:

```python
timer = SamplingTimer(frequency_hz=10)
```

The only input is the sampling frequency.

---

# 📤 **Outputs**

The timer does not return values.  
It controls timing by sleeping until the next tick.

---

# 🔍 **Internal Logic**

### **1. Interval Calculation**
```python
self.interval = 1.0 / frequency_hz
```

### **2. Drift Correction**
If the loop is late:

```python
if now > self.next_tick:
    self.next_tick = now + self.interval
```

This prevents drift accumulation.

### **3. Sleep Until Next Tick**
```python
sleep_time = self.next_tick - now
time.sleep(sleep_time)
```

### **4. Schedule Next Tick**
```python
self.next_tick += self.interval
```

---

# 🔗 **Integration with `generator.py`**

In your generator loop, replace:

```python
time.sleep(sleep_interval)
```

with:

```python
self.timer.sleep_until_next_tick()
```

Specifically:

```python
from ..utils.sampling_timer import SamplingTimer

self.timer = SamplingTimer(config["frequency_hz"])
```

Then inside `_run_loop()`:

```python
self.timer.sleep_until_next_tick()
```

This gives you **stable, real‑time sampling**.

---

# 🧩 **Why this design is strong**

- **High precision** using `time.perf_counter()`  
- **Drift‑free**  
- **Thread‑safe**  
- **No Qt dependency**  
- **Minimal overhead**  
- **Scales to high frequencies (1–10,000 Hz)**  

This is exactly what a telemetry generator needs.

---


## 4.3. preview_buffer.py

Here is the final utility module for Project A’s GUI layer: a clean, minimal, high‑performance **`preview_buffer.py`**.  
This module is intentionally simple and robust: it maintains a rolling buffer of recent preview samples so the GUI can display smooth, real‑time telemetry without storing huge amounts of data.

---

# 📄 `preview_buffer.py`  
### **Full Implementation (Rolling buffer for live preview)**

```python
# generator/utils/preview_buffer.py

from collections import deque
from typing import Dict, Any, List


class PreviewBuffer:
    """
    Rolling buffer for storing recent preview samples.

    Purpose:
        - Store the last N preview samples for each column
        - Provide fast append and fast retrieval
        - Avoid unbounded memory growth
        - Decouple preview storage from the GUI and generator

    Parameters:
        max_points: int
            Maximum number of samples to keep per column
    """

    def __init__(self, max_points: int = 500):
        self.max_points = max_points
        self.buffers: Dict[str, deque] = {}

    # ---------------------------------------------------------
    # Public API
    # ---------------------------------------------------------
    def add_sample(self, sample: Dict[str, Any]):
        """
        Adds a new sample to the buffer.

        Args:
            sample: dict
                Example:
                    {
                        "Temperature": 42.1,
                        "Motor RPM": 1500,
                        "Voltage": 230.5
                    }
        """
        for col, value in sample.items():
            if col not in self.buffers:
                self.buffers[col] = deque(maxlen=self.max_points)
            self.buffers[col].append(value)

    def get_series(self, column: str) -> List[Any]:
        """
        Returns the rolling series for a given column.

        Args:
            column: str
                Column name

        Returns:
            list of recent values (up to max_points)
        """
        if column not in self.buffers:
            return []
        return list(self.buffers[column])

    def clear(self):
        """
        Clears all buffers.
        """
        self.buffers.clear()
```

---

# 🧠 **Detailed Explanation of `preview_buffer.py`**

This module provides a **rolling memory** for preview data.  
It is intentionally decoupled from both the GUI and the generator so that:

- the generator can push preview samples without worrying about GUI state  
- the preview panel can pull data at its own pace  
- memory usage stays constant (bounded by `max_points`)  

It is the simplest and most robust architecture for real‑time preview.

---

# 🎯 Purpose

The `PreviewBuffer`:

- stores the last *N* samples for each column  
- ensures constant memory usage  
- provides fast append and fast retrieval  
- supports multiple columns simultaneously  
- allows the preview panel to render smooth rolling plots  

It is the “data backbone” of the live preview system.

---

# 📥 Inputs

From the generator:

```python
preview_buffer.add_sample({"Temperature": 42.1, "Motor RPM": 1500})
```

Each sample is a dictionary mapping column names to values.

---

# 📤 Outputs

The preview panel retrieves data via:

```python
series = preview_buffer.get_series("Temperature")
```

This returns a list of up to `max_points` values.

---

# 🔍 Internal Logic

### **1. Buffers per column**
```python
self.buffers: Dict[str, deque] = {}
```

Each column gets its own `deque(maxlen=N)`.

### **2. Append new samples**
```python
self.buffers[col].append(value)
```

### **3. Retrieve rolling series**
```python
return list(self.buffers[column])
```

### **4. Clear all buffers**
Useful when restarting generation.

---

# 🔗 Integration with the rest of Project A

### In `TelemetryGenerator`  
You can optionally maintain a buffer:

```python
self.preview_buffer.add_sample(last_row)
self.preview_callback(last_row)
```

### In `PreviewPanel`  
You can use the buffer to render smoother plots:

```python
y = preview_buffer.get_series(self.current_column)
```

This is optional — your current design uses direct callbacks, which is also fine.  
The buffer simply gives you more flexibility.

---

# 🧩 Why this design is strong

- **Constant memory usage**  
- **Fast** (deque is O(1) append/pop)  
- **Thread‑safe for append‑only patterns**  
- **Decoupled** from GUI and generator  
- **Extensible** (multi‑column preview, multi‑plot preview, etc.)  

It’s exactly the right tool for real‑time telemetry visualization.

---


# 5. app.py

Let us close the loop for Project A with a clean **entry point**.

Below is a minimal, production‑ready **`app.py`** that wires everything together and launches the Generator GUI.

---

### 📄 `app.py`

```python
# generator/app.py

import sys
from PySide6.QtWidgets import QApplication
from gui.main_window import MainWindow


def main():
    """
    Entry point for the Telemetry Generator GUI.

    Responsibilities:
        - Create the Qt application
        - Instantiate MainWindow
        - Show the GUI
        - Start the Qt event loop
    """
    app = QApplication(sys.argv)

    window = MainWindow()
    window.show()

    sys.exit(app.exec())


if __name__ == "__main__":
    main()
```

---

### Purpose

- **`main()`** creates the `QApplication`, instantiates `MainWindow`, shows it, and starts the event loop.
- This is the only file you need to run directly, e.g.:

```bash
python -m generator.app
```

or from the project root:

```bash
python generator/app.py
```


# 6. Start the application

## 6.1. Install python packages

Here is a **clean, unified pip‑install list** we can paste directly into a Jupyter Notebook cell to install *all* required packages for **both Project A (Generator)** and **Project B (Analyzer)**.

This list is fully consistent with the lightweight architecture we built:  
no NLTK, no spaCy, no Torch, no SHAP — everything runs fast and clean.

---

# ✅ **Pip install list for Jupyter Notebook (Projects A + B)**

We can paste this directly into a Jupyter cell:

```python
!pip install numpy pandas scikit-learn pyarrow PySide6 matplotlib scipy tqdm
```

This installs:

| Package | Purpose |
|--------|---------|
| **numpy** | Core numerical operations |
| **pandas** | DataFrames, CSV/Parquet handling |
| **scikit-learn** | Clustering, forecasting, anomaly detection, XAI |
| **pyarrow** | Parquet read/write for Generator + Analyzer |
| **PySide6** | GUI framework for Analyzer |
| **matplotlib** | Plotting inside the Analyzer |
| **scipy** | Optional FFT and numerical utilities |
| **tqdm** | Progress bars for Generator |

---

# 🧠 Why this list is complete

Because of our design choices:

- **NLP** uses regex + Counter → no NLTK/spaCy  
- **Deep Learning** uses IsolationForest → no PyTorch/TensorFlow  
- **XAI** uses RandomForestRegressor → no SHAP  
- **Forecasting** uses LinearRegression → no statsmodels/Prophet  

This keeps the environment:

- lightweight  
- fast  
- easy to install  
- fully compatible with Jupyter  

---


In [ ]:
!pip install numpy pandas scikit-learn pyarrow PySide6 matplotlib scipy tqdm

## 6.2. Run instructions

Let’s make this really concrete and reproducible: we are in a Jupyter Notebook, and we want to run **Project A (Generator)** and **Project B (Analyzer)** from there, using the codebase we’ve built.

I’ll assume a layout like:

```text
project-root/
  generator/
    app.py
    ...
  analyzer/
    app.py
    ...
  config.json
```

We’re running Jupyter with the working directory set to `project-root`.

---

## 1. Install all dependencies from the notebook

In a fresh notebook cell:

```python
!pip install numpy pandas scikit-learn pyarrow PySide6 matplotlib scipy tqdm
```

Restart the kernel after installation if needed.

---

## 2. Make sure Python can see your packages

If `generator` and `analyzer` are plain folders (no installation as packages), add the project root to `sys.path`:

```python
import sys, os

project_root = os.path.abspath(".")  # or explicit path
if project_root not in sys.path:
    sys.path.append(project_root)
```

Now `import generator` and `import analyzer` will work.

---

## 3. Running Project A (Generator) from Jupyter

You have two main options:

### 3.1. Run as a script via `!python`

If `generator/app.py` is the entry point:

```python
!python -m generator.app
```

If `app.py` expects arguments (e.g., output file, config path), you can pass them:

```python
!python -m generator.app --config config.json
```

This runs the Generator as a normal script from inside the notebook.  
It will:

- read/write `config.json`  
- generate `telemetry.parquet` (or CSV)  
- possibly run until completion or in a loop, depending on your implementation.

### 3.2. Run by importing and calling `main()`

If `generator/app.py` exposes a `main()` function:

```python
from generator.app import main as generator_main

generator_main()
```

This keeps everything in the same Python process.  
If the Generator is long‑running (e.g., infinite stream), you might want to run it in a background thread or a separate terminal instead—Jupyter will block until it finishes.

---

## 4. Running Project B (Analyzer) from Jupyter

The Analyzer is a **Qt GUI application** (PySide6). That means:

- it starts its own event loop (`app.exec()`)  
- it opens a separate window  
- it should ideally run in its own process, not inside the Jupyter kernel’s event loop

So the cleanest way from Jupyter is to call it as a script.

### 4.1. Run Analyzer as a script via `!python`

Assuming `analyzer/app.py` is the entry point we defined:

```python
!python -m analyzer.app
```

If you want to pass a specific `config.json`:

```python
!python -m analyzer.app config.json
```

This will:

- start a separate process  
- open the Analyzer GUI window  
- read `config.json`  
- start `AlertListener`, `AnalyzerLoop`, etc.

Your Jupyter kernel remains free.

> This is the most robust way: Jupyter for orchestration, Analyzer in its own process.

### 4.2. Import and call `main()` (possible, but less ideal)

You *can* do:

```python
from analyzer.app import main as analyzer_main

analyzer_main()
```

But:

- the Qt event loop will block the notebook cell until the GUI is closed  
- mixing Qt event loops and Jupyter’s event loop can be fragile  

I’d only use this for quick tests, not for regular work.

---

## 5. End‑to‑end workflow from a single notebook

Here’s a realistic, reproducible flow.

### 5.1. Setup cell

```python
import sys, os

project_root = os.path.abspath(".")
if project_root not in sys.path:
    sys.path.append(project_root)

!pip install numpy pandas scikit-learn pyarrow PySide6 matplotlib scipy tqdm
```

Restart kernel if needed, then re‑run the `sys.path` cell.

### 5.2. Generate telemetry with Project A

Option A — run once, generate a finite file:

```python
!python -m generator.app
```

Option B — if you have a `main()`:

```python
from generator.app import main as generator_main

generator_main()  # blocks until done
```

After this, you should have:

- `config.json`  
- `telemetry.parquet` (or CSV)  

in your project root (or configured paths).

### 5.3. Start Analyzer (Project B) to inspect the data

In a new cell:

```python
!python -m analyzer.app config.json
```

This opens the GUI:

- reads `config.json`  
- loads `telemetry.parquet`  
- runs all selected modules (statistics, clustering, forecasting, NLP, deep_learning, xai)  
- updates plots and health indicators  

You can keep the Analyzer running while you:

- regenerate data with the Generator  
- tweak parameters in the notebook  
- re‑run the Generator to append more telemetry  

If your Generator sends socket alerts, the Analyzer will react in real time.

---

## 6. Advanced: orchestrating both from Jupyter

If you want to orchestrate both from one notebook, you can:

- run the Analyzer in a separate process  
- run the Generator in another cell (or another process)  
- use `config.json` + socket alerts as the contract

Example:

```python
# Cell 1: start Analyzer (non-blocking, in background shell)
!python -m analyzer.app config.json &
```

```python
# Cell 2: run Generator
!python -m generator.app
```

(Backgrounding with `&` works in some Jupyter setups; in others you might prefer a separate terminal.)

---


With our folder structure:

```
D:\DigitalTwinsGeneratorGUI\
    analyzer\
        app.py
        ...
    generator\
        app.py
        ...
    config.json
```

we can run **both Project A and Project B directly from a single Jupyter Notebook** using a clean, reproducible runner notebook.

Below is a complete, ready‑to‑use **Jupyter Notebook runner** that:

- sets up the environment  
- adds our project to `sys.path`  
- runs **Generator** (Project A)  
- runs **Analyzer** (Project B GUI)  
- supports background execution  
- works on Windows (your setup)  

---

# 📘 **Jupyter Notebook Runner for Project A + B**

Create a new notebook in:

```
D:\DigitalTwinsGeneratorGUI\run.ipynb
```

and paste the following cells.

---

# 🟦 **Cell 1 — Setup environment & paths**

```python
import sys
import os

# Path to your main folder
PROJECT_ROOT = r"D:\DigitalTwinsGeneratorGUI"

# Add to sys.path so Python can import generator and analyzer
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Project root added to sys.path:", PROJECT_ROOT)
```

This ensures:

```python
from generator.app import main
from analyzer.app import main
```

works correctly.

---

# 🟦 **Cell 2 — Install all required packages**

```python
!pip install numpy pandas scikit-learn pyarrow PySide6 matplotlib scipy tqdm
```

Restart the kernel once after installation.

---

# 🟦 **Cell 3 — Run Project A (Generator)**

You have two clean options.

---

## **Option A — Run Generator as a separate process (recommended)**  
This keeps Jupyter responsive.

```python
!python D:\DigitalTwinsGeneratorGUI\generator\app.py
```

If your generator accepts a config path:

```python
!python D:\DigitalTwinsGeneratorGUI\generator\app.py D:\DigitalTwinsGeneratorGUI\config.json
```

---

## **Option B — Run Generator inside the notebook (blocks until done)**

```python
from generator.app import main as generator_main

generator_main()
```

Use this only if the generator finishes quickly.  
If it streams indefinitely, prefer Option A.

---

# 🟦 **Cell 4 — Run Project B (Analyzer GUI)**

The Analyzer is a **PySide6 Qt GUI**, so it must run in its own process.  
Running it inside the notebook would block the kernel and conflict with Jupyter’s event loop.

So we run it as a separate process:

```python
!python D:\DigitalTwinsGeneratorGUI\analyzer\app.py D:\DigitalTwinsGeneratorGUI\config.json
```

This will:

- open the Analyzer GUI window  
- load `config.json`  
- start the alert listener  
- start the analysis loop  
- update plots in real time  

Our Jupyter Notebook remains free to run other cells.

---

# 🟦 **Cell 5 — (Optional) Run Analyzer in background**

If you want the GUI to start but keep the notebook cell free immediately:

```python
import subprocess

analyzer_process = subprocess.Popen(
    ["python", r"D:\DigitalTwinsGeneratorGUI\analyzer\app.py", r"D:\DigitalTwinsGeneratorGUI\config.json"]
)

print("Analyzer started with PID:", analyzer_process.pid)
```

You can later stop it:

```python
analyzer_process.terminate()
```

---

# 🟦 **Cell 6 — (Optional) Regenerate telemetry while Analyzer is running**

If the Analyzer is open and listening for alerts, you can regenerate data:

```python
!python D:\DigitalTwinsGeneratorGUI\generator\app.py
```

The Analyzer will:

- detect file changes  
- receive socket alerts  
- refresh plots  
- update health indicators  

This gives us a full **real‑time digital twin loop** from inside Jupyter.

---

# 🎯 **Summary: How to run both projects from Jupyter**

| Task | Best Method | Why |
|------|-------------|------|
| Run Generator | `!python generator/app.py` | Non‑blocking, clean |
| Run Analyzer | `!python analyzer/app.py` | Required for Qt GUI |
| Run both simultaneously | Analyzer in background + Generator in foreground | Real‑time loop |
| Import and run inside notebook | Only for Generator | Analyzer must run in separate process |

---
